# Molt Voice Service — Kaggle GPU starter

Run the cells from top to bottom. This notebook clones the API repo, installs its dependencies, starts the service, then creates a temporary public URL.

**Before running:** set a GPU accelerator and enable Internet in Kaggle Notebook options. Attach a Kaggle Dataset containing your reference voice WAV, and add `GROQ_API_KEY` or `DEEPSEEK_API_KEY` under Notebook → Add-ons → Secrets. The URL is public and temporary; use this for testing, not production.

In [ ]:
# Edit these values before Run all. VOICE_REF_TEXT must exactly match the attached WAV.
REPO_URL = "https://github.com/crackerghost/molt-voice-service.git"
REPO_BRANCH = "main"
VOICE_REF_AUDIO = "/kaggle/input/your-dataset-name/my_voice.wav"
VOICE_REF_TEXT = "Paste the exact transcript of your reference voice clip here."
VOICE_PORT = 8000

# Example dataset path: /kaggle/input/my-voice-data/my_voice.wav

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

voice_ref = Path(VOICE_REF_AUDIO)
if not voice_ref.is_file():
    wav_candidates = sorted(Path("/kaggle/input").rglob("*.wav"))
    if len(wav_candidates) == 1:
        voice_ref = wav_candidates[0]
        print(f"Auto-detected reference WAV: {voice_ref}")
    else:
        raise FileNotFoundError(
            f"Set VOICE_REF_AUDIO to a Kaggle input WAV. Found: {wav_candidates}"
        )
if not voice_ref.is_file():
    raise FileNotFoundError(f"Set VOICE_REF_AUDIO to the attached Kaggle Dataset WAV. Not found: {voice_ref}")
if not VOICE_REF_TEXT or VOICE_REF_TEXT.startswith("Paste the exact"):
    raise ValueError("Set VOICE_REF_TEXT to the exact spoken words in the reference WAV.")

repo_dir = Path("/kaggle/working/molt-voice-service")
if not (repo_dir / "server" / "__main__.py").is_file():
    subprocess.run(["git", "clone", "--depth=1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)], check=True)

print("Installing service requirements; this can take several minutes on a fresh Kaggle session.")
# RapidOCR is optional and has no release compatible with Kaggle's Python 3.13.
# Keep the rest of the service requirements; screen OCR will disable itself if absent.
requirements = (repo_dir / "requirements.txt").read_text().splitlines()
kaggle_requirements = Path("/kaggle/working/requirements-kaggle.txt")
kaggle_requirements.write_text("\n".join(
    line for line in requirements
    if not line.strip().lower().startswith("rapidocr-onnxruntime")
) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(kaggle_requirements)], check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator in Notebook options, then rerun this cell.")

## Load Kaggle secrets and start the API

The API needs an LLM provider key for `/ready` and chat. The key is read from Kaggle Secrets; it is not embedded in the notebook. The voice WAV and its exact transcript are also required for voice cloning.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def read_secret(name):
    try:
        return secrets.get_secret(name).strip()
    except Exception:
        return ""

groq_key = read_secret("GROQ_API_KEY")
deepseek_key = read_secret("DEEPSEEK_API_KEY")
if not groq_key and not deepseek_key:
    raise RuntimeError("Add GROQ_API_KEY or DEEPSEEK_API_KEY in Kaggle Notebook Secrets, then rerun this cell.")

os.environ.update({
    "VOICE_HOST": "127.0.0.1",
    "VOICE_PORT": str(VOICE_PORT),
    "VOICE_REF_AUDIO": str(voice_ref.resolve()),
    "VOICE_REF_TEXT": VOICE_REF_TEXT,
    "VOICE_API_DEVICE": "cuda",
    "VOICE_API_DTYPE": "fp16",
    "VOICE_NUM_STEP": "6",
    "VOICE_FIRST_STEP": "5",
    "ASR_BACKEND": "faster-whisper",
    "ASR_MODEL": "large-v3-turbo",
    "ASR_DEVICE": "cuda",
    "ASR_COMPUTE": "float16",
    "HF_HUB_DISABLE_XET": "1",
    "VOICE_KILL_EXISTING": "0",
})
if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
    os.environ["LLM_PROVIDER"] = "groq"
else:
    os.environ["DEEPSEEK_API_KEY"] = deepseek_key
    os.environ["LLM_PROVIDER"] = "deepseek"

from IPython.display import display, Markdown
import json, time, urllib.request

def get_json(url):
    with urllib.request.urlopen(url, timeout=10) as response:
        return json.loads(response.read().decode("utf-8"))

if "voice_proc" not in globals() or voice_proc.poll() is not None:
    api_log = open("/kaggle/working/molt-voice-api.log", "w")
    voice_proc = subprocess.Popen([sys.executable, "-m", "server"], cwd=repo_dir, env=os.environ.copy(), stdout=api_log, stderr=subprocess.STDOUT, start_new_session=True)
else:
    print("Voice API process is already running in this notebook.")

base_url = f"http://127.0.0.1:{VOICE_PORT}"
deadline = time.time() + 1800
while time.time() < deadline:
    if voice_proc.poll() is not None:
        print(Path("/kaggle/working/molt-voice-api.log").read_text(errors="replace")[-6000:])
        raise RuntimeError("Voice API exited before becoming healthy; inspect the startup log above.")
    try:
        health = get_json(base_url + "/health")
        break
    except Exception:
        time.sleep(5)
else:
    raise TimeoutError("Voice API did not become healthy in 30 minutes. Check /kaggle/working/molt-voice-api.log.")

print("Voice API health:", health)
print("Readiness:", get_json(base_url + "/ready"))

## Create the temporary public URL

This uses Cloudflare Quick Tunnels, which need no account or domain. The URL is ephemeral and publicly reachable; anyone with it can call the service. Keep it private and stop the session when finished.

In [ ]:
import re
import urllib.request

cloudflared = Path("/kaggle/working/cloudflared")
if not cloudflared.exists():
    download_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(download_url, cloudflared)
    cloudflared.chmod(0o755)

if "tunnel_proc" not in globals() or tunnel_proc.poll() is not None:
    tunnel_log = open("/kaggle/working/cloudflared.log", "w")
    tunnel_proc = subprocess.Popen([str(cloudflared), "tunnel", "--no-autoupdate", "--url", base_url], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
else:
    print("Cloudflare tunnel is already running in this notebook.")

tunnel_log_path = Path("/kaggle/working/cloudflared.log")
tunnel_url = None
deadline = time.time() + 120
while time.time() < deadline:
    if tunnel_proc.poll() is not None:
        raise RuntimeError(tunnel_log_path.read_text(errors="replace")[-6000:])
    log_text = tunnel_log_path.read_text(errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(2)
if not tunnel_url:
    raise TimeoutError("No trycloudflare.com URL appeared; check the cloudflared log.")

display(Markdown(f"## Molt Voice API URL\n\n`{tunnel_url}`\n\nHealth: `{tunnel_url}/health`  ·  Readiness: `{tunnel_url}/ready`\n\nSet `VITE_VOICE_API_URL={tunnel_url}` in the consuming frontend. Keep this notebook session running."))

## Stop the service when finished

Run this cell to close the tunnel and stop the API before ending your Kaggle session.

In [ ]:
for proc_name in ("tunnel_proc", "voice_proc"):
    proc = globals().get(proc_name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"Stopped {proc_name}.")